## 11.3 Combining and Differencing
### 11.3.1 Normalized Difference Vegetation Index

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import pandas as pd
from skimage.exposure import adjust_gamma, rescale_intensity
from sklearn.metrics import DistanceMetric
import xarray as xr

In [ ]:
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 12],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

In [ ]:
filename_C03 = "OR_ABI-L1b-RadM1-M6C03_G16_s20242642035250_e20242642035308_c20242642035343.nc"
abi_path = Path("./data/abi") / filename_C03
abi_C03 = xr.open_dataset(abi_path, engine="h5netcdf")
veggie = abi_C03["Rad"].values

filename_C02 = "OR_ABI-L1b-RadM1-M6C02_G16_s20242642035250_e20242642035308_c20242642035336.nc"
abi_path = Path("./data/abi") / filename_C02
abi_C02 = xr.open_dataset(abi_path, engine="h5netcdf")
red = abi_C02["Rad"].values

In [ ]:
red = red[::2, ::2]

In [ ]:
ndvi = (veggie - red) / (veggie + red)

In [ ]:
abi_C02.nominal_satellite_subpoint_lon.item()

In [ ]:
def get_sat_params(abi):
    geom = abi.goes_imager_projection
    satellite_params = {
        "central_lon": geom.longitude_of_projection_origin,
        "central_lat": 0.0,
        "sat_height": geom.perspective_point_height,
        "semi_major_axis" : geom.semi_major_axis,
        "semi_minor_axis" : geom.semi_minor_axis,
        "R_earth" : 6371000
    }

    return satellite_params


def get_geo_crs(abi):
    abi_params = get_sat_params(abi)
    globe = ccrs.Globe(semimajor_axis=abi_params["semi_major_axis"], \
        semiminor_axis=abi_params["semi_minor_axis"])
    geo_crs = ccrs.Geostationary(central_longitude=abi_params["central_lon"], \
        satellite_height=abi_params["sat_height"], globe=globe)
    return geo_crs


def get_extent(abi):
    abi_params = get_sat_params(abi)
    X = abi["x"] * abi_params["sat_height"]
    Y = abi["y"] * abi_params["sat_height"]
    return (X.min(), X.max(), Y.min(), Y.max())


In [ ]:
geo_crs_g16 = get_geo_crs(abi_C03)
extent_eastusa = get_extent(abi_C03)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g16)
ax.coastlines("10m", color="white")

im = plt.imshow(ndvi, vmin=-0.5, vmax=0.5, cmap="BrBG", \
    extent=extent_eastusa)
plt.colorbar(im, orientation="horizontal", pad=0.05, \
    label="NDVI", shrink=0.75)   

ax.set_extent(extent_eastusa, crs=geo_crs_g16)
plt.show()

In [ ]:
filename_ACCM = "OR_ABI-L2-ACMM1-M6_G16_s20242642035250_e20242642035308_c20242642035485.nc"
abi_path = Path("./data/acm") / filename_ACCM
abi_ACCM = xr.open_dataset(abi_path, engine="h5netcdf")
cloud_mask = abi_ACCM["BCM"]

In [ ]:
cloud_mask_hires = np.repeat(np.repeat(cloud_mask, 2, axis=1), 2, axis=0)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g16)
ax.coastlines("10m", color="red")

im = plt.imshow(cloud_mask_hires, vmin=0, vmax=1, cmap="binary", extent=extent_eastusa)
cbar = plt.colorbar(im, orientation="horizontal", pad=0.05, 
        shrink=0.75, boundaries=[-0.5, 0.5, 1.5], ticks=[0, 1])
ax.set_extent(extent_eastusa, crs=geo_crs_g16)

cbar.set_ticklabels(["Clear", "Cloudy"])
plt.show()

In [ ]:
ndvi_masked = ma.masked_array(ndvi, \
    mask=cloud_mask_hires, fill_value=np.nan)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g16)
ax.coastlines("10m", color="white")

im = plt.imshow(ma.filled(ndvi_masked), vmin=-0.5, vmax=0.5, 
    cmap="BrBG", extent=extent_eastusa)
plt.colorbar(im, orientation="horizontal", pad=0.05, 
    label="NDVI", shrink=0.75)

ax.set_extent(extent_eastusa, crs=geo_crs_g16)
ax.set_facecolor("black")

plt.show()

### 11.3.2 Window Channels

In [ ]:
filename = "OR_ABI-L1b-RadM2-M6C13_G19_s20252981300559_e20252981301029_c20252981301078.nc"
abi_path = Path("./data/abi") / filename
abi_C13 = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C13 = abi_C13["Rad"].values

filename = "OR_ABI-L1b-RadM2-M6C02_G19_s20252981300559_e20252981301017_c20252981301044.nc"
abi_path = Path("./data/abi") / filename
abi_C02 = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C02 = abi_C02["Rad"].values[::4,::4]

In [ ]:
sandwich = abi_rad_C02 - abi_rad_C13

In [ ]:
geo_crs_g19 = get_geo_crs(abi_C13)
extent_carribean = get_extent(abi_C13)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g19)
ax.coastlines("10m", color="black")
im = plt.imshow(sandwich, cmap="terrain", extent=extent_carribean)
plt.colorbar(im, orientation="horizontal", pad=0.05, \
    label="Sandwich (Channel 2 - Channel 13)", shrink=0.75)
ax.set_extent(extent_carribean, crs=geo_crs_g19)

plt.show()

## 11.4 RGB

### 11.4.1 True Color

In [ ]:
filename = "OR_ABI-L1b-RadM2-M6C01_G18_s20250072000550_e20250072001008_c20250072001040.nc"
abi_path = Path("./data/abi") / filename
abi_C01_socal = xr.open_dataset(abi_path, engine="h5netcdf")
abi_C01_rad_socal = abi_C01_socal["Rad"].values

filename = "OR_ABI-L1b-RadM2-M6C02_G18_s20250072000550_e20250072001008_c20250072001034.nc"
abi_path = Path("./data/abi") / filename
abi_C02_socal = xr.open_dataset(abi_path, engine="h5netcdf")
abi_C02_rad_socal = abi_C02_socal["Rad"].values

filename = "OR_ABI-L1b-RadM2-M6C03_G18_s20250072000550_e20250072001009_c20250072001047.nc"
abi_path = Path("./data/abi") / filename
abi_C03_socal = xr.open_dataset(abi_path, engine="h5netcdf")
abi_C03_rad_socal = abi_C03_socal["Rad"].values

In [ ]:
geo_crs_g18 = get_geo_crs(abi_C01_socal)
extent_socal = get_extent(abi_C01_socal)

In [ ]:
def scale_and_gamma(img, gamma=1.0, out_range=(0, 1)):
    img = rescale_intensity(img, out_range=out_range)
    img = adjust_gamma(img, gamma)
    return img

In [ ]:
blue_socal_adj = scale_and_gamma(abi_C01_rad_socal, gamma=0.5)
veggie_socal_adj = scale_and_gamma(abi_C02_rad_socal[::2,::2], gamma=0.5)
red_socal_adj = scale_and_gamma(abi_C03_rad_socal, gamma=0.5)

Fractional Combination from https://agupubs-onlinelibrary-wiley-com.proxy-um.researchport.umd.edu/doi/10.1029/2018EA000379

Green = 0.45 * Red + 0.10 * NIR + 0.45 * Blue

In [ ]:
# https://github.com/pytroll/satpy/blob/579e797091af6bfaa30b0923d4735fe4e402b502/satpy/composites/abi.py
green_socal_adj = (0.45 * red_socal_adj + 0.45 * blue_socal_adj + \
    0.1 * veggie_socal_adj)

In [ ]:
rgb_tc = np.stack([red_socal_adj, green_socal_adj, blue_socal_adj], axis=2)

In [ ]:
rgb_lims = [0, 0.75]

fig, axes = plt.subplots(ncols=3, figsize=[15, 5], \
     subplot_kw={"projection": geo_crs_g18})


for ax in axes:
    ax.set_extent(extent_socal, crs=geo_crs_g18)
    ax.coastlines("10m", color="black")

axes[0].set_title("Red")
axes[0].imshow(red_socal_adj, vmin=rgb_lims[0], vmax=rgb_lims[1], \
    cmap="Reds", origin="upper", \
    extent=extent_socal, transform=geo_crs_g18)

axes[1].set_title("Green")
axes[1].imshow(green_socal_adj, vmin=rgb_lims[0], vmax=rgb_lims[1], \
    cmap="Greens", origin="upper", \
    extent=extent_socal, transform=geo_crs_g18)

axes[2].set_title("Blue")
axes[2].imshow(blue_socal_adj, vmin=rgb_lims[0], vmax=rgb_lims[1], \
    cmap="Blues", origin="upper", \
    extent=extent_socal, transform=geo_crs_g18)

plt.show()

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g18)
ax.coastlines("10m", color="white")

im = plt.imshow(rgb_tc, extent=extent_socal)

ax.set_extent(extent_socal, crs=geo_crs_g18)
plt.show()

### 11.4.2 Dust RGB

Credit to Aerosol Watch for this example: https://bsky.app/profile/aerosolwatch.bsky.social/post/3lkqfgzpwtk27

In [ ]:
filename_C11 = "OR_ABI-L1b-RadM1-M6C11_G16_s20250780900277_e20250780900335_c20250780900381.nc"
abi_path = Path("./data/abi") / filename_C11
abi_C11_swusa = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C11_swusa = abi_C11_swusa["Rad"].values

filename_C13 = "OR_ABI-L1b-RadM1-M6C13_G16_s20250780900277_e20250780900347_c20250780900399.nc"
abi_path = Path("./data/abi") / filename_C13
abi_C13_swusa = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C13_swusa = abi_C13_swusa["Rad"].values

filename_C14 = "OR_ABI-L1b-RadM1-M6C14_G16_s20250780900277_e20250780900335_c20250780900387.nc"
abi_path = Path("./data/abi") / filename_C14
abi_C14_swusa = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C14_swusa = abi_C14_swusa["Rad"].values

filename_C15 = "OR_ABI-L1b-RadM1-M6C15_G16_s20250780900277_e20250780900341_c20250780900390.nc"
abi_path = Path("./data/abi") / filename_C15
abi_C15_swusa = xr.open_dataset(abi_path, engine="h5netcdf")
abi_rad_C15_swusa = abi_C15_swusa["Rad"].values

In [ ]:
extent_swusa = get_extent(abi_C15_swusa)

In [ ]:
red_dust_adj = scale_and_gamma(abi_rad_C15_swusa - abi_rad_C13_swusa)
green_dust_adj = scale_and_gamma(abi_rad_C14_swusa - abi_rad_C11_swusa, gamma=2.5)
blue_dust_adj = scale_and_gamma(abi_rad_C13_swusa)

In [ ]:
rgb_dust = np.stack([red_dust_adj, green_dust_adj, blue_dust_adj], axis=2)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g16)
ax.add_feature(cfeature.STATES, edgecolor="white")

plt.imshow(rgb_dust, extent=extent_swusa)

ax.set_extent(extent_swusa, crs=geo_crs_g16)

plt.show()

### 11.4.3 Fire/Natural RGB

In [ ]:
filename_C05 = "OR_ABI-L1b-RadM2-M6C05_G18_s20250072000550_e20250072001008_c20250072001043.nc"
abi_path = Path("./data/abi") / filename_C05
abi_C05_socal = xr.open_dataset(abi_path, engine="h5netcdf")
abi_C05_rad_socal = abi_C05_socal["Rad"].values

In [ ]:
red_natrgb = scale_and_gamma(abi_C05_rad_socal)
green_natrgb = scale_and_gamma(abi_C03_rad_socal)
blue_natrgb = scale_and_gamma(abi_C02_rad_socal[::2, ::2])

In [ ]:
rgb_natrgb = np.stack([red_natrgb, green_natrgb, blue_natrgb], axis=2)

In [ ]:
plt.figure()
ax = plt.subplot(projection=geo_crs_g18)
ax.add_feature(cfeature.STATES, edgecolor="white")

plt.imshow(rgb_natrgb, extent=extent_socal)

ax.set_extent(extent_socal, crs=geo_crs_g18)

plt.show()


## 11.5 Matching with Surface Observations

In [ ]:
filename_DSR = "OR_ABI-L2-DSRM1-M6_G16_s20192091300534_e20192091300591_c20192091303116.nc"
dsr_path = Path("./data/dsr") / filename_DSR
goes_dsr = xr.open_dataset(dsr_path, engine="h5netcdf")
dsr = goes_dsr["DSR"].values
dsr_lon = goes_dsr["lon"].values
dsr_lat = goes_dsr["lat"].values

In [ ]:
x_dsr, y_dsr = np.meshgrid(dsr_lon, dsr_lat)

In [ ]:
station_loc = (-77.93, 40.72)
extent_eusa_dsr = [-80, -70, 34, 45]

plt.figure()

ax=plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.STATES, edgecolor="black")
ax.set_extent(extent_eusa_dsr, crs=ccrs.PlateCarree())

im = plt.pcolormesh(x_dsr, y_dsr, dsr, transform=ccrs.PlateCarree())
plt.colorbar(im, orientation="horizontal", pad=0.05, \
    label="Downward Shortwave Radiation (W m-2)", shrink=0.75)
plt.scatter(*station_loc, marker="x", c="black", s=200)

plt.show()

### 11.5.1 With user-defined functions

In [ ]:
def haversine(deglon1, deglat1, deglon2, deglat2):

    r_earth = 6378.0
    
    lat1 = np.radians(deglat1)
    lat2 = np.radians(deglat2)
    
    long1 = np.radians(deglon1)
    long2 = np.radians(deglon2)
      
    a = np.sin(0.5*(lat2-lat1))
    b = np.sin(0.5*(long2-long1))
    
    dist = r_earth*2.0*np.arcsin( \
        np.sqrt(a*a+np.cos(lat1)*np.cos(lat2)*b*b))
    
    return dist

In [ ]:
def matchup_spatial(longitude, latitude, site_lon, site_lat,
        radius_km=50.0, closest_only=False):
    
    distance_matrix = np.full(latitude.shape,  6378.0)

    dist_deg = np.sqrt((np.array(latitude)-site_lat)**2 \
        + (np.array(longitude)-site_lon)**2)
    close_pts = (dist_deg < 1.0)
    
    distance_matrix[close_pts] = haversine(site_lon, site_lat, \
        longitude[close_pts], latitude[close_pts])
    keep_index = (distance_matrix > radius_km)
    
    if closest_only:
       if len(keep_index[keep_index==True]) > 0:
           keep_index = (distance_matrix == distance_matrix.min())
   
    return keep_index

In [ ]:
dsr_mask = matchup_spatial(x_dsr, y_dsr, *station_loc, radius_km=100.0)

In [ ]:
dsr[dsr_mask]

In [ ]:
dsr_ma = np.ma.masked_array(dsr, mask=dsr_mask)
x_ma = np.ma.masked_array(x_dsr, mask=dsr_mask)
y_ma = np.ma.masked_array(y_dsr, mask=dsr_mask)

In [ ]:
plt.figure(figsize=[8,8])
ax=plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.STATES, edgecolor="black")
ax.set_extent(extent_eusa_dsr, crs=ccrs.PlateCarree())

im = plt.pcolormesh(x_ma, y_ma, dsr_ma, transform=ccrs.PlateCarree())

plt.colorbar(im, orientation="horizontal", pad=0.05, \
    label="Downward Shortwave Radiation (W m-2)", shrink=0.75)

plt.scatter(*station_loc, marker="x", c="black", s=100)

ax.set_aspect("equal")

plt.show()

In [ ]:
filename_header = "surfrad_header.txt"
header_path = Path("./data/txt") / filename_header
header = pd.read_csv(header_path)

filename_surfrad = "psu19209.dat"
surfrad_path = Path("./data/txt") / filename_surfrad
insitu_dsr = pd.read_csv(surfrad_path, \
    skipinitialspace=True, sep='\\s+', \
    skiprows=2, header=None, names=list(header))

In [ ]:
surfrad_df = pd.DataFrame({
    "year": insitu_dsr["year"], 
    "month": insitu_dsr["month"],
    "day": insitu_dsr["day"], 
    "hour": insitu_dsr["hour"], 
    "minute" : insitu_dsr["min"]
        })

In [ ]:
goes_dsr_time = goes_dsr.time_coverage_start[0:19]
goes_dsr_time

In [ ]:
insitu_dsr["Datetime"] = pd.to_datetime(surfrad_df)

fmt = "%Y-%m-%dT%H:%M:%S"
dsr_time = pd.to_datetime(goes_dsr_time, \
    format=fmt)

In [ ]:
def matchup_temporal(time, time_array, matchup_max_time_mins=15):
    time_diff = [np.abs(x - time) for x in time_array]
    index = np.argmin(time_diff)
    
    delta = abs(time_array[index] - time)

    if delta < pd.Timedelta(minutes=matchup_max_time_mins):
        return index
    else:
        return -1

In [ ]:
index = matchup_temporal(dsr_time, insitu_dsr["Datetime"] )
print(index)

In [ ]:
insitu_dsr["dw_solar"][index] - dsr_ma.mean()

### 11.5.2 With Machine Learning

In [ ]:
station_coords = [[station_loc[1], station_loc[0]]]
grid_coords = [[i, j] for i in dsr_lat.flat  for j in dsr_lon.flat]

In [ ]:
dist = DistanceMetric.get_metric("haversine")

In [ ]:
def deg_to_rad(pairs):
    return [[np.radians(x[0]), np.radians(x[1])] \
        for x in pairs]

station_coords_rad = deg_to_rad(station_coords)
grid_coords_rad = deg_to_rad(grid_coords)

In [ ]:
r_earth = 6356.0  # in km
distances = dist.pairwise(station_coords_rad, grid_coords_rad).flatten()*r_earth
distances_2d = distances.reshape(dsr.shape)

In [ ]:
mask_sk2d = distances_2d > 100

dsr_ma_sk = np.ma.masked_array(dsr, mask=mask_sk2d)
x_ma_sk = np.ma.masked_array(x_dsr, mask=mask_sk2d)
y_ma_sk = np.ma.masked_array(y_dsr, mask=mask_sk2d)

In [ ]:
proj = dict(projection=ccrs.PlateCarree())
fig, axes = plt.subplots(ncols = 2, figsize=[8,12], subplot_kw=proj)

for axis in axes:
    axis.add_feature(cfeature.STATES, edgecolor="black")
    axis.set_extent(extent_eusa_dsr, crs=ccrs.PlateCarree())
    axis.scatter(*station_loc, marker="x", c="red", s=100, zorder=3)

axes[0].set_title("Distance from Station")
left = axes[0].pcolormesh(x_dsr, y_dsr, distances_2d, \
    vmin=0, vmax=900, transform=ccrs.PlateCarree())
plt.colorbar(left, orientation="horizontal", pad=0.05, \
    label="Distance (km)", shrink=0.75)

axes[1].set_title("Matches < 100 km")
right = axes[1].pcolormesh(x_ma_sk, y_ma_sk, dsr_ma_sk, \
    transform=ccrs.PlateCarree())
plt.colorbar(right, orientation="horizontal", pad=0.05, \
    label="DSR (W m-2)", shrink=0.75)
    
plt.show()